## Tahap 0 (lanjutan) — RSA check pakai sel demografi IRISAN asli

Follow-up dari `notebooks/06_tahap0_rsa_kaggle.ipynb` (lihat
`notes/research_question/03_pivot2_group_consistency.md` §12.10). Tes
sebelumnya cuma pakai kelompok **1-atribut** (mis. `"RACE: Asian"` doang)
sebagai proxy -- ini ulangi metodologi yang sama pakai sel **irisan** asli
(mis. `"Black | Hindu"`), target sebenarnya dari `L_group` (contoh motivasi
awal Thread 2).

Data sel irisan dibangun dari respons individual mentah Pew ATP (15 wave)
lewat `scripts/build_intersectional_cells.py` -- **169 sel** (raw label
unik cuma 163, tapi 6 di antaranya kebetulan bentrok teks antar tipe
kombinasi berbeda, mis. `"Other | Other"` bisa muncul dari RACExPOLPARTY
maupun RELIGxPOLPARTY -- makanya dibedakan pakai `attribute + group`, bukan
`group` doang), 6 kombinasi atribut (`RACExRELIG`, `RACExPOLPARTY`,
`RACExPOLIDEOLOGY`, `RELIGxPOLPARTY`, `EDUCATIONxINCOME`, `AGExPOLPARTY`),
ambang minimal 30 responden per sel per gelombang survei.

**Yang beda dari tes sebelumnya:** dulu cuma 2 jenis pasangan (sesama-atribut
vs lintas-atribut). Sekarang sel-nya punya 2 komponen, jadi ada **3 jenis
pasangan**:

1. **Beda 1 komponen doang** (mis. `"Black | Hindu"` vs `"Black | Muslim"` --
   ras sama, agama beda) -- ini yang **paling relevan** buat kasus nyata
   `L_group` (cari tetangga buat sel yang datanya sedikit).
2. **Beda 2 komponen sekaligus, tapi masih 1 tipe kombinasi** (mis.
   `"Black | Hindu"` vs `"White | Muslim"`, sama-sama RACExRELIG).
3. **Beda tipe kombinasi sama sekali** (mis. RACExRELIG vs
   EDUCATIONxINCOME) -- analog "lintas-atribut" dari tes sebelumnya,
   prioritas rendah.

## Sebelum jalan: setting Kaggle

1. **Accelerator**: GPU T4 x2 atau P100. **Internet: On**.
2. **Upload** `opinionqa_intersectional.csv`
   (path asli: `datasets/subpop/data/opinionqa/processed/opinionqa_intersectional.csv`)
   sebagai Kaggle Dataset, attach ke notebook ini -- caranya sama seperti
   notebook 06 (Add Data -> Upload -> Create).

Perkiraan waktu: 15-25 menit.

In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy scikit-learn tqdm
!pip install -q -U bitsandbytes

In [ ]:
import os
import ast
import glob
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, spearmanr
from sklearn.metrics import pairwise_distances
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    raise RuntimeError(
        "GPU tidak terdeteksi. Cek Notebook options -> Accelerator -> GPU T4 x2/P100, "
        "restart & Run All lagi."
    )

In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"
USE_4BIT = False

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError(
        "Tidak ketemu opinionqa_intersectional.csv. Upload dulu sebagai Kaggle Dataset, "
        "attach, lalu jalankan ulang cell ini. Atau isi manual: DATA_PATH = '/kaggle/input/.../opinionqa_intersectional.csv'"
    )
print("Pakai data dari:", DATA_PATH)

N_QKEYS_FOR_WD = 300  # sampel pertanyaan bersama buat hitung jarak-asli tiap pasang sel (biar cepat)
RANDOM_SEED = 42
N_PERMUTATIONS = 2000

OUT_DIR = "/kaggle/working/tahap0_rsa_intersectional"
assert not OUT_DIR.startswith("/kaggle/input"), "OUT_DIR harus di /kaggle/working, bukan /kaggle/input!"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

In [ ]:
df = pd.read_csv(DATA_PATH)

def _parse_list(x):
    return ast.literal_eval(x) if isinstance(x, str) else x

df["responses"] = df["responses"].apply(_parse_list)
df["ordinal"] = df["ordinal"].apply(_parse_list)
df["options"] = df["options"].apply(_parse_list)
df["group_key"] = df["attribute"] + " :: " + df["group"]

print(f"Total baris: {len(df)}")
print(f"Jumlah sel irisan unik: {df['group_key'].nunique()}")
print(f"Jumlah pertanyaan (qkey) unik: {df['qkey'].nunique()}")
print(f"Kombinasi atribut: {sorted(df['attribute'].unique().tolist())}")

## 1. Siapkan metadata sel: pisahkan jadi komponen (tipe kombinasi, nilai-1, nilai-2)

Dipakai buat bikin 3 masker pasangan di bawah.

In [ ]:
GROUP_KEYS = sorted(df["group_key"].unique().tolist())
n_g = len(GROUP_KEYS)
print(f"{n_g} sel irisan")

group_meta = {}
for gk in GROUP_KEYS:
    attr_type, group_str = gk.split(" :: ", 1)
    v1, v2 = group_str.split(" | ", 1)
    group_meta[gk] = {"attr_type": attr_type, "v1": v1, "v2": v2}

attr_types = np.array([group_meta[gk]["attr_type"] for gk in GROUP_KEYS])
v1_arr = np.array([group_meta[gk]["v1"] for gk in GROUP_KEYS])
v2_arr = np.array([group_meta[gk]["v2"] for gk in GROUP_KEYS])

same_type = attr_types[:, None] == attr_types[None, :]
share_v1 = v1_arr[:, None] == v1_arr[None, :]
share_v2 = v2_arr[:, None] == v2_arr[None, :]
share_exactly_one = same_type & (share_v1 ^ share_v2)
share_neither_same_type = same_type & (~share_v1) & (~share_v2)
diff_type = ~same_type

print("Jumlah pasangan per jenis (dari total", n_g * (n_g - 1) // 2, "pasangan):")
print("  beda 1 komponen doang       :", np.triu(share_exactly_one, k=1).sum())
print("  beda 2 komponen, tipe sama  :", np.triu(share_neither_same_type, k=1).sum())
print("  beda tipe kombinasi         :", np.triu(diff_type, k=1).sum())

## 2. Hitung jarak-asli antar sel (Wasserstein Distance, disampel biar cepat)

In [ ]:
resp_lookup = {
    (gk, qk): (resp, ordv)
    for gk, qk, resp, ordv in zip(df["group_key"], df["qkey"], df["responses"], df["ordinal"])
}
qkeys_by_group = df.groupby("group_key")["qkey"].apply(set).to_dict()

rng = np.random.default_rng(RANDOM_SEED)
group_real_dist = np.full((n_g, n_g), np.nan)

for i in tqdm(range(n_g), desc="Hitung jarak-asli antar sel irisan"):
    gi = GROUP_KEYS[i]
    for j in range(i, n_g):
        if i == j:
            group_real_dist[i, j] = 0.0
            continue
        gj = GROUP_KEYS[j]
        shared = qkeys_by_group.get(gi, set()) & qkeys_by_group.get(gj, set())
        if not shared:
            continue
        shared = list(shared)
        if len(shared) > N_QKEYS_FOR_WD:
            idx = rng.choice(len(shared), size=N_QKEYS_FOR_WD, replace=False)
            shared = [shared[k] for k in idx]

        wds = []
        for qk in shared:
            respA, ordA = resp_lookup[(gi, qk)]
            respB, ordB = resp_lookup[(gj, qk)]
            if len(ordA) != len(ordB):
                continue
            wds.append(wasserstein_distance(ordA, ordB, u_weights=respA, v_weights=respB))
        if wds:
            group_real_dist[i, j] = group_real_dist[j, i] = float(np.mean(wds))

n_pairs_total = n_g * (n_g - 1) // 2
n_missing = int(np.isnan(group_real_dist[np.triu_indices(n_g, k=1)]).sum())
print(f"Selesai. Pasangan tanpa pertanyaan bersama (jarak-asli = NaN): {n_missing} dari {n_pairs_total}")
print("\nContoh 6 sel paling mirip sama sel pertama:")
print(pd.Series(group_real_dist[0], index=GROUP_KEYS).sort_values().head(7))

## 3. Bangun prompt kalimat natural per sel irisan

Format kalimat beda per tipe kombinasi (biar natural), tapi semuanya
tanpa menyebut kode atribut internal survei -- pelajaran dari
`06_tahap0_rsa_kaggle.ipynb` (format `"ATTR: value"` sudah dicek bukan
biang keroknya, tapi kalimat natural tetap best practice).

In [ ]:
def fmt_polideology(v):
    return v.lower()

PAIR_TEMPLATES = {
    "RACExRELIG": lambda v1, v2: f"This survey respondent's race is {v1} and their religion is {v2}.",
    "RACExPOLPARTY": lambda v1, v2: f"This survey respondent's race is {v1} and their political party affiliation is {v2}.",
    "RACExPOLIDEOLOGY": lambda v1, v2: f"This survey respondent's race is {v1}, and politically they describe their views as {fmt_polideology(v2)}.",
    "RELIGxPOLPARTY": lambda v1, v2: f"This survey respondent's religion is {v1} and their political party affiliation is {v2}.",
    "EDUCATIONxINCOME": lambda v1, v2: f"This survey respondent's highest level of education is {v1}, and their household income is {v2}.",
    "AGExPOLPARTY": lambda v1, v2: f"This survey respondent is {v1} years old and their political party affiliation is {v2}.",
}

def build_prompt(gk):
    meta = group_meta[gk]
    return PAIR_TEMPLATES[meta["attr_type"]](meta["v1"], meta["v2"])

group_prompts = {gk: build_prompt(gk) for gk in GROUP_KEYS}
print("Contoh prompt per tipe kombinasi:\n")
seen_types = set()
for gk in GROUP_KEYS:
    t = group_meta[gk]["attr_type"]
    if t not in seen_types:
        print(f"[{t}] {group_prompts[gk]!r}")
        seen_types.add(t)

## 4. Load model & ekstrak representasi

In [ ]:
print(f"Loading tokenizer & model: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = dict(torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True)
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs.pop("torch_dtype", None)
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **model_kwargs)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
print(f"Model loaded. Jumlah layer: {NUM_LAYERS} (+1 embedding awal)")

In [ ]:
@torch.no_grad()
def get_hidden_states_all_layers(prompt: str) -> np.ndarray:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs, output_hidden_states=True)
    hs = torch.stack(out.hidden_states, dim=0)
    return hs[:, 0, -1, :].float().cpu().numpy()

group_embeddings = {}
for gk, prompt in tqdm(group_prompts.items(), desc="Ekstraksi representasi sel irisan"):
    group_embeddings[gk] = get_hidden_states_all_layers(prompt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

group_emb_array = np.stack([group_embeddings[gk] for gk in GROUP_KEYS])
n_layers_total = group_emb_array.shape[1]

np.savez(
    os.path.join(OUT_DIR, "embeddings_intersectional.npz"),
    group_emb=group_emb_array,
    group_keys=np.array(GROUP_KEYS, dtype=object),
)
print("Representasi disimpan ke", os.path.join(OUT_DIR, "embeddings_intersectional.npz"))

## 5. RSA per jenis pasangan, per layer

Sama seperti sebelumnya: korelasi Spearman antara jarak-representasi-LLM vs
jarak-asli, plus permutation test (Mantel test) di layer terbaik.

In [ ]:
def upper_tri(mat):
    idx = np.triu_indices_from(mat, k=1)
    return mat[idx]

def rho_masked(emb_array, real_dist_matrix, layer_idx, pair_mask):
    rep_dist = pairwise_distances(emb_array[:, layer_idx, :], metric="cosine")
    idx = np.triu_indices_from(real_dist_matrix, k=1)
    keep = pair_mask[idx]
    real_flat, rep_flat = real_dist_matrix[idx][keep], rep_dist[idx][keep]
    valid = ~np.isnan(real_flat) & ~np.isnan(rep_flat)
    if valid.sum() < 5:
        return np.nan
    rho, _ = spearmanr(real_flat[valid], rep_flat[valid])
    return rho

def rsa_perm_masked(emb_array, real_dist_matrix, layer_idx, pair_mask, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED):
    rep_dist = pairwise_distances(emb_array[:, layer_idx, :], metric="cosine")
    idx = np.triu_indices_from(real_dist_matrix, k=1)
    keep = pair_mask[idx]
    real_flat, rep_flat = real_dist_matrix[idx][keep], rep_dist[idx][keep]
    valid = ~np.isnan(real_flat) & ~np.isnan(rep_flat)
    real_flat, rep_flat = real_flat[valid], rep_flat[valid]
    rho, _ = spearmanr(real_flat, rep_flat)

    rng_p = np.random.default_rng(seed)
    n = real_dist_matrix.shape[0]
    perm_rhos = np.empty(n_perm)
    for k in range(n_perm):
        perm = rng_p.permutation(n)
        permuted_flat = real_dist_matrix[np.ix_(perm, perm)][idx][keep][valid]
        r, _ = spearmanr(permuted_flat, rep_flat)
        perm_rhos[k] = 0.0 if np.isnan(r) else r
    return float(rho), float(np.mean(np.abs(perm_rhos) >= abs(rho)))

MASKS = {
    "beda_1_komponen": share_exactly_one,
    "beda_2_komponen_tipe_sama": share_neither_same_type,
    "beda_tipe_kombinasi": diff_type,
}

rho_table = {"layer": list(range(n_layers_total))}
for name, mask in MASKS.items():
    rho_table[name] = [rho_masked(group_emb_array, group_real_dist, L, mask) for L in range(n_layers_total)]

rho_df = pd.DataFrame(rho_table)
print(rho_df.to_string())

In [ ]:
summary_rows = []
for name, mask in MASKS.items():
    rhos = rho_df[name].values
    best_layer = int(np.nanargmax(rhos))  # cari yang PALING POSITIF, bukan abs -- kita nyari arah yang benar
    rho_best, p_best = rsa_perm_masked(group_emb_array, group_real_dist, best_layer, mask)
    summary_rows.append({"jenis_pasangan": name, "best_layer": best_layer, "rho": rho_best, "p_value": p_best,
                          "n_pairs": int(np.triu(mask, k=1).sum())})
    print(f"[{name}] layer terbaik {best_layer}: rho={rho_best:.3f}, p={p_best:.4f} (n={int(np.triu(mask, k=1).sum())} pasang)")

summary_df = pd.DataFrame(summary_rows)

In [ ]:
print("--- breakdown 'beda_1_komponen' PER TIPE KOMBINASI (bukan pooled 6 tipe) ---")
print("(rho pooled 719 pasang bisa nutupin 1 tipe lemah kalau tipe lain kuat -- cek RACExRELIG sendiri)")
for t in sorted(set(attr_types)):
    type_mask = share_exactly_one & (attr_types[:, None] == t) & (attr_types[None, :] == t)
    n_pairs = int(np.triu(type_mask, k=1).sum())
    if n_pairs < 5:
        print(f"{t}: cuma {n_pairs} pasang, skip")
        continue
    rhos = [rho_masked(group_emb_array, group_real_dist, L, type_mask) for L in range(n_layers_total)]
    best_layer = int(np.nanargmax(rhos))
    rho_best, p_best = rsa_perm_masked(group_emb_array, group_real_dist, best_layer, type_mask)
    print(f"{t}: layer {best_layer}, rho={rho_best:.3f}, p={p_best:.4f}, n={n_pairs}")

## 5b. Pendalaman geometri embedding (track: perdalam embedding)

RSA di atas cuma ngasih tau *ranking*-nya lumayan (rho ~0.17), TAPI nggak
ngasih tau **seberapa lebar** sebaran embedding-nya. Bisa aja rho positif
padahal semua sel mepet (rho itu rank-based, cuek sama jarak absolut). Tiga
cek di bawah (semua murah, cuma numpy di atas `group_emb_array` yang udah ada):

- **Step 1 — sebaran jarak:** min/median/max cosine distance antar sel di
  layer terbaik. Jawab langsung: "jangan-jangan mirip semua?"
- **Step 2 — anisotropi + mean-centering:** ada arah dominan bersama yang
  nutupin sinyal? Kalau habis di-center rho-nya NAIK → sinyal cuma ketutup,
  bisa digali. Kalau tetap → ya emang segitu.
- **Step 3 — per-tipe:** udah ada di cell atas (RACExRELIG diisolasi).

In [ ]:
# === STEP 1: seberapa lebar sebaran jarak antar sel? (layer terbaik beda_1) ===
BEST = int(summary_df.loc[summary_df["jenis_pasangan"] == "beda_1_komponen", "best_layer"].iloc[0])
emb_best = group_emb_array[:, BEST, :]
d = pairwise_distances(emb_best, metric="cosine")
iu = np.triu_indices(n_g, k=1)
alld = d[iu]

print(f"Layer terbaik (beda_1_komponen) = {BEST}. Sebaran cosine distance antar {n_g} sel:\n")
print(f"  SEMUA {len(alld)} pasang : min={alld.min():.4f}  median={np.median(alld):.4f}  "
      f"max={alld.max():.4f}  std={alld.std():.4f}")
print(f"  rasio sebaran (max-min)/mean = {(alld.max()-alld.min())/alld.mean():.3f}")
print(f"  -> cosine SIMILARITY: pasangan paling BEDA pun masih {1-alld.max():.3f} mirip")

# pisah per jenis pasangan biar keliatan yang tipe-sama vs beda-tipe
for name, mask in MASKS.items():
    sub = d[iu][mask[iu]]
    sub = sub[~np.isnan(sub)]
    if len(sub):
        print(f"  [{name}] min={sub.min():.4f} median={np.median(sub):.4f} max={sub.max():.4f} (n={len(sub)})")

print("\n  Cara baca: kalau max jauh lebih kecil dari 1 (mis <0.1) & rasio sebaran kecil,")
print("  berarti semua sel numpuk di 1 kerucut sempit -> 'mirip semua' beneran kejadian.")

fig, ax = plt.subplots(figsize=(7, 4))
for name, mask in MASKS.items():
    sub = d[iu][mask[iu]]
    sub = sub[~np.isnan(sub)]
    if len(sub):
        ax.hist(sub, bins=40, alpha=0.55, label=f"{name} (n={len(sub)})")
ax.set_xlabel(f"cosine distance antar sel (layer {BEST})")
ax.set_ylabel("jumlah pasang")
ax.set_title(f"Step 1 -- sebaran jarak embedding antar sel (layer {BEST})")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "step1_sebaran_jarak.png"), dpi=150)
plt.show()

In [ ]:
# === STEP 2: anisotropi (arah dominan bersama) + efek mean-centering ===

# 2a. Seberapa "searah" semua embedding di layer terbaik?
mean_vec = emb_best.mean(axis=0)
norm_mean = np.linalg.norm(mean_vec)
mean_norm = np.linalg.norm(emb_best, axis=1).mean()
print(f"=== STEP 2a: anisotropi di layer {BEST} ===")
print(f"  ||rata-rata vektor|| / rata-rata ||vektor|| = {norm_mean/mean_norm:.3f}")
print("  (deket 1 = semua vektor nunjuk 1 arah yang sama = anisotropik parah;")
print("   deket 0 = arahnya nyebar ke mana-mana)")

# PCA pada embedding yang SUDAH di-center: sisa variasinya low-rank atau nggak?
Xc = emb_best - mean_vec
s = np.linalg.svd(Xc, compute_uv=False)
evr = (s**2) / (s**2).sum()
print(f"\n  Setelah di-center, variasi sisa dimakan tiap komponen (PCA):")
print(f"  PC1={evr[0]:.1%}, PC2={evr[1]:.1%}, PC3={evr[2]:.1%}, "
      f"5 teratas total={evr[:5].sum():.1%}")

# 2b. INTI: apakah mean-centering NAIKIN rho? (sinyal ketutup arah bersama?)
# center per-layer (buang rata-rata antar sel di tiap layer), lalu ukur ulang RSA.
group_emb_centered = group_emb_array - group_emb_array.mean(axis=0, keepdims=True)

print(f"\n=== STEP 2b: rho SEBELUM vs SESUDAH mean-centering (beda_1_komponen) ===")
print(f"  {'layer':>5} | {'rho asli':>9} | {'rho center':>10} | delta")
best_gain = (None, -9)
for L in range(n_layers_total):
    r0 = rho_masked(group_emb_array, group_real_dist, L, share_exactly_one)
    r1 = rho_masked(group_emb_centered, group_real_dist, L, share_exactly_one)
    if not np.isnan(r0):
        tag = "  <-- naik" if (r1 - r0) > 0.02 else ""
        print(f"  {L:>5} | {r0:>+9.3f} | {r1:>+10.3f} | {r1-r0:>+.3f}{tag}")
        if r1 > best_gain[1]:
            best_gain = (L, r1)

print(f"\n  rho tertinggi setelah centering: layer {best_gain[0]}, rho={best_gain[1]:+.3f}")
print("  -> kalau kolom 'rho center' konsisten LEBIH TINGGI dari 'rho asli', berarti")
print("     sinyal demografis emang ketutup arah dominan bersama -- mean-centering")
print("     (atau whitening) layak dipakai sebelum bikin kernel. Kalau nggak naik,")
print("     berarti masalahnya bukan di situ, cari di tempat lain (prompt/pooling).")

## 5c. Gabung temuan: per-tipe SETELAH centering + precision@k

Dua cek lanjutan (nyambungin finding 02 & 03):

- **Step 4 — per-tipe × centering:** apakah mean-centering nyelametin
  RACExRELIG yang lemah? Apakah AGE/EDU tembus lebih tinggi dari 0.50?
- **Step 5 — precision@k:** rho global itu ranking SEMUA pasang, tapi kernel
  `L_group` cuma pakai k tetangga terdekat. Cek langsung: dari k tetangga
  terdekat menurut embedding, berapa yang beneran k-terdekat menurut survei?
  Dibandingin sama chance (tebak acak) biar angkanya berarti.

In [ ]:
# === STEP 4: per-tipe, rho SEBELUM vs SESUDAH mean-centering (beda_1_komponen) ===
import warnings
warnings.filterwarnings("ignore")  # spearmanr ConstantInputWarning di subset kecil, nggak penting

print(f"{'tipe':<18} | {'rho asli':>8} @L | {'rho center':>10} @L | n pasang")
print("-" * 60)
for t in sorted(set(attr_types)):
    type_mask = share_exactly_one & (attr_types[:, None] == t) & (attr_types[None, :] == t)
    n_pairs = int(np.triu(type_mask, k=1).sum())
    if n_pairs < 5:
        print(f"{t:<18} | cuma {n_pairs} pasang, skip")
        continue
    rr = [rho_masked(group_emb_array, group_real_dist, L, type_mask) for L in range(n_layers_total)]
    rc = [rho_masked(group_emb_centered, group_real_dist, L, type_mask) for L in range(n_layers_total)]
    Lr, Lc = int(np.nanargmax(rr)), int(np.nanargmax(rc))
    print(f"{t:<18} | {rr[Lr]:>+8.3f} {Lr:>2} | {rc[Lc]:>+10.3f} {Lc:>2} | {n_pairs}")

print("\n  -> lihat khususnya RACExRELIG: kalau kolom center-nya lompat jauh, berarti")
print("     sinyalnya cuma ketutup (bukan nggak ada). Kalau tetap rendah, ya emang lemah.")

In [ ]:
# === STEP 5: precision@k -- top-k tetangga menurut embedding, akurat nggak? ===
K = 5

def precision_at_k(emb_layer, subset, k=K):
    """subset = index sel dari 1 tipe. Untuk tiap sel: dari k-terdekat menurut
    embedding, berapa yang juga k-terdekat menurut survei? Rata-rata over sel."""
    sub = np.array(subset)
    ed = pairwise_distances(emb_layer[sub], metric="cosine")
    rd = group_real_dist[np.ix_(sub, sub)].copy()
    np.fill_diagonal(ed, np.inf)
    np.fill_diagonal(rd, np.inf)
    precs = []
    for i in range(len(sub)):
        valid = ~np.isnan(rd[i])
        if valid.sum() < k:
            continue
        emb_nn = set(np.argsort(ed[i])[:k])
        surv_nn = set(np.argsort(np.where(valid, rd[i], np.inf))[:k])
        precs.append(len(emb_nn & surv_nn) / k)
    return (float(np.mean(precs)), len(precs)) if precs else (np.nan, 0)

print(f"precision@{K} (dalam-tipe): berapa dari {K} tetangga-terdekat-embedding")
print(f"yang beneran {K}-terdekat menurut survei. Bandingin raw vs center vs chance.\n")
print(f"{'tipe':<18} | {'raw':>6} | {'center':>6} | {'chance':>6} | m sel")
print("-" * 55)
for t in sorted(set(attr_types)):
    subset = [i for i in range(n_g) if attr_types[i] == t]
    m = len(subset)
    if m < K + 2:
        print(f"{t:<18} | cuma {m} sel, skip")
        continue
    # pilih layer terbaik tipe ini (versi centered), pakai layer sama buat raw
    tm = share_exactly_one & (attr_types[:, None] == t) & (attr_types[None, :] == t)
    rc = [rho_masked(group_emb_centered, group_real_dist, L, tm) for L in range(n_layers_total)]
    Lc = int(np.nanargmax(rc))
    pr_raw, _ = precision_at_k(group_emb_array[:, Lc, :], subset)
    pr_cen, nsel = precision_at_k(group_emb_centered[:, Lc, :], subset)
    chance = K / (m - 1)
    print(f"{t:<18} | {pr_raw:>6.2f} | {pr_cen:>6.2f} | {chance:>6.2f} | {m}")

print(f"\n  -> 'center' jauh DI ATAS 'chance' = kernel top-{K} beneran informatif")
print(f"     (ini yang paling penting buat L_group, lebih dari rho global).")
print(f"     'center' cuma setara 'chance' = walau rho oke, tetangga top-{K} nggak kepake.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name in MASKS:
    ax.plot(rho_df["layer"], rho_df[name], marker="o", label=name)
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(15, color="red", linestyle="--", alpha=0.4, label="layer ~15 (ambang llm-opinions)")
ax.set_xlabel("Layer")
ax.set_ylabel("RSA Spearman rho")
ax.set_title("Tahap 0 (sel irisan) -- RSA per jenis pasangan, per layer")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "rsa_intersectional_per_layer.png"), dpi=150)
plt.show()

rho_df.to_csv(os.path.join(OUT_DIR, "rsa_intersectional_rho_per_layer.csv"), index=False)
summary_df.to_csv(os.path.join(OUT_DIR, "rsa_intersectional_summary.csv"), index=False)
print(summary_df.to_string(index=False))
print(f"\nOutput ada di: {OUT_DIR}")

## 6. Lihat langsung: Peta 1 (jarak asli) vs Peta 2 (jarak di kepala LLM)

Ini inti Tahap 0 -- bukan cuma angka rho, tapi beneran **cocokkan dua peta
jarak** buat pasangan yang sama:

- **Peta 1** = jarak asli dari data survei (`group_real_dist`, sudah
  dihitung di bagian 2).
- **Peta 2** = jarak representasi LLM (cosine distance dari embedding, di
  layer terbaik yang ketemu buat `beda_1_komponen`).

Tiap titik di scatter plot di bawah = **1 pasang sel**. Sumbu X = Peta 1,
sumbu Y = Peta 2. Kalau titik-titiknya condong naik dari kiri-bawah ke
kanan-atas (garis miring ke atas), berarti dua peta itu nyambung (ini yang
diringkas jadi rho positif). Kalau berantakan/nggak ada arah, berarti nggak
nyambung.

In [ ]:
BEST_LAYER_FOR_PLOT = int(summary_df.loc[summary_df["jenis_pasangan"] == "beda_1_komponen", "best_layer"].iloc[0])
rep_dist_at_best = pairwise_distances(group_emb_array[:, BEST_LAYER_FOR_PLOT, :], metric="cosine")

colors = {
    "beda_1_komponen": "#1f77b4",
    "beda_2_komponen_tipe_sama": "#ff7f0e",
    "beda_tipe_kombinasi": "#7f7f7f",
}
idx_ut = np.triu_indices(n_g, k=1)

fig, ax = plt.subplots(figsize=(7, 6))
for name, mask in MASKS.items():
    keep = mask[idx_ut]
    x = group_real_dist[idx_ut][keep]
    y = rep_dist_at_best[idx_ut][keep]
    valid = ~np.isnan(x) & ~np.isnan(y)
    alpha = 0.35 if name == "beda_tipe_kombinasi" else 0.6
    ax.scatter(x[valid], y[valid], s=10, alpha=alpha, label=f"{name} (n={int(valid.sum())})", color=colors[name])

ax.set_xlabel("Peta 1: jarak asli dari survei (Wasserstein Distance)")
ax.set_ylabel(f"Peta 2: jarak representasi LLM (cosine, layer {BEST_LAYER_FOR_PLOT})")
ax.set_title("Tahap 0 (sel irisan) -- Peta 1 vs Peta 2, tiap titik = 1 pasang sel")
ax.legend(markerscale=2, fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "rsa_scatter_peta1_vs_peta2.png"), dpi=150)
plt.show()

Biar makin konkret, ini beberapa pasang sel **nyata** dari data kita
(bukan hipotetis) -- Peta 1 dan Peta 2 diprint berdampingan:

In [ ]:
def show_pair(gk1, gk2, label):
    if gk1 not in GROUP_KEYS or gk2 not in GROUP_KEYS:
        print(f"[{label}] salah satu sel nggak ada di data ({gk1} / {gk2}), skip.\n")
        return
    i, j = GROUP_KEYS.index(gk1), GROUP_KEYS.index(gk2)
    print(f"[{label}]")
    print(f"  {gk1}")
    print(f"  {gk2}")
    print(f"  Peta 1 (jarak asli, WD)                     = {group_real_dist[i, j]:.4f}")
    print(f"  Peta 2 (jarak representasi LLM, layer {BEST_LAYER_FOR_PLOT}) = {rep_dist_at_best[i, j]:.4f}")
    print()

show_pair("RACExRELIG :: Asian | Hindu", "RACExRELIG :: Asian | Protestant",
          "beda 1 komponen -- RACE sama (Asian), RELIG beda")
show_pair("RACExRELIG :: White | Protestant", "RACExRELIG :: Black | Protestant",
          "beda 1 komponen -- RELIG sama (Protestant), RACE beda")
show_pair("RACExRELIG :: Asian | Hindu", "RACExRELIG :: White | Atheist",
          "beda 2 komponen, tipe sama (RACE beda DAN RELIG beda)")
show_pair("RACExRELIG :: Asian | Hindu", "EDUCATIONxINCOME :: Associate's degree | $50,000-$75,000",
          "beda tipe kombinasi sama sekali")

## Cara baca hasilnya

Yang paling penting: baris **`beda_1_komponen`** -- ini yang paling mirip
sama kasus nyata `L_group` (cari tetangga buat sel yang datanya sedikit,
beda di 1 ciri doang). Kalau rho-nya positif & signifikan, itu konfirmasi
kuat kalau temuan di tes 1-atribut sebelumnya (§12.10) memang berlaku juga
buat sel irisan asli, bukan cuma kebetulan struktur data proxy-nya.

Scatter plot & contoh pasangan konkret di bagian 6 itu **bukti visual
langsung** dari angka rho itu -- kalau titik biru (`beda_1_komponen`)
kelihatan condong naik dari kiri-bawah ke kanan-atas dibanding titik abu-abu
(`beda_tipe_kombinasi`) yang berantakan/condong turun, itu cara paling
konkret buat lihat "dua peta ini nyambung apa nggak" tanpa perlu percaya
angka rho begitu saja.

**Langkah selanjutnya:** download `/kaggle/working/tahap0_rsa_intersectional/`
(sekarang termasuk `rsa_scatter_peta1_vs_peta2.png`), bandingkan angkanya
sama tabel di `notes/research_question/03_pivot2_group_consistency.md`
§12.10/§12.11, lalu update dokumen itu kalau ada temuan baru dari scatter
plot/contoh konkretnya.